### <div style="background-color: #e7f3fe; border-left: 6px solid #2196F3; padding: 15px; border-radius: 4px; color: #0c5460;"> <strong>Imports</strong>
</div>.

In [3]:
import sys, os
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, classification_report, recall_score
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.config import BATCH_SIZE, DEVICE, SAMPLE_FRAC
from src.dataset import SkinCancerDataset, get_transforms
from src.model import build_model

In [4]:
datadir = PROJECT_ROOT / "data/raw"
metadata_path = datadir / "HAM10000_metadata.csv"

df = pd.read_csv(metadata_path)

image_dir_1 = datadir / "HAM10000_images_part_1"
image_dir_2 = datadir / "HAM10000_images_part_2"

image_path_dict = {
    os.path.splitext(f)[0]: str((image_dir_1 / f).relative_to(PROJECT_ROOT))
    for f in os.listdir(image_dir_1)
}

image_path_dict.update({
    os.path.splitext(f)[0]: str((image_dir_2 / f).relative_to(PROJECT_ROOT))
    for f in os.listdir(image_dir_2)
})

df["path"] = df["image_id"].map(image_path_dict)

In [5]:
df.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,data\raw\HAM10000_images_part_1\ISIC_0027419.jpg
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,data\raw\HAM10000_images_part_1\ISIC_0025030.jpg
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,data\raw\HAM10000_images_part_1\ISIC_0026769.jpg
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,data\raw\HAM10000_images_part_1\ISIC_0025661.jpg
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,data\raw\HAM10000_images_part_2\ISIC_0031633.jpg


In [6]:
df.columns

Index(['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization',
       'path'],
      dtype='object')

In [7]:
dataset = SkinCancerDataset(
    df=df,
    transform=get_transforms(),
    sample_frac=SAMPLE_FRAC,
    image_path_col="path"
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

images, labels = next(iter(loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

Image batch shape: torch.Size([16, 3, 160, 160])
Label batch shape: torch.Size([16])
Labels: tensor([5, 4, 5, 2, 5, 2, 5, 5, 5, 5])


In [8]:
print(images.shape)

torch.Size([16, 3, 160, 160])


In [9]:
model = build_model()
model = model.to(DEVICE)

outputs = model(images)

print("Output shape:", outputs.shape)

Output shape: torch.Size([16, 7])


In [10]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [11]:
loss = criterion(outputs, labels)
print("Loss:", loss.item())

Loss: 2.333867311477661


In [12]:
model.train()

for epoch in range(1):  # just 1 epoch for now
    total_loss = 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        # Forward pass
        outputs = model(images)

        # Loss
        loss = criterion(outputs, labels)

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 1, Loss: 1.0168


In [13]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
macro_recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)

# melanoma class = 4 based on our label map
melanoma_recall = recall_score(
    all_labels,
    all_preds,
    labels=[4],
    average="macro",
    zero_division=0
)

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro Recall: {macro_recall:.4f}")
print(f"Melanoma Recall: {melanoma_recall:.4f}")

Accuracy: 0.6467
Macro Recall: 0.1519
Melanoma Recall: 0.0000


In [14]:
checkpoint_dir = PROJECT_ROOT / "outputs/checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / "local_poc_mobilenetv3_small.pt"

torch.save(model.state_dict(), checkpoint_path)

print(f"Saved checkpoint to: {checkpoint_path}")

Saved checkpoint to: C:\Users\binia\OneDrive\Bini\Education - Professional Development\Springboard\git\healthCare\skincancer-pytorch\outputs\checkpoints\local_poc_mobilenetv3_small.pt
